# Demand Forecasting Analytical Report

This notebook documents EDA, de-censoring, backtesting, baseline comparison, and leakage validation.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt

from src.data_loading import load_data, validate_schema
from src.preprocessing import clean_data, sort_time_series, add_stockout_flag
from src.features import build_features
from src.decensoring import create_demand_proxy, summarize_censoring
from src.training import train_and_evaluate
from src.validation import evaluate_baselines, future_permutation_test, get_three_month_backtest_split

ModuleNotFoundError: No module named 'src'

## 1. Load Data

In [ ]:
# Update the path if your CSV file has a different name
DATA_PATH = '../data/demand_train_val_1.5years.csv'

df = load_data(DATA_PATH)
validate_schema(df)
df.head()

## 2. Exploratory Data Analysis

In [ ]:
display(df.info())
display(df.isna().sum())
display(df.describe())

In [ ]:
df.groupby(df['timestamp'].dt.date)['sales'].sum().plot(figsize=(12, 4), title='Daily Sales Trend')
plt.ylabel('Sales')
plt.tight_layout()
plt.show()

## 3. Preprocessing, Feature Engineering and De-censoring

In [ ]:
df_clean = clean_data(df)
df_clean = sort_time_series(df_clean)
df_clean = add_stockout_flag(df_clean)
df_features = build_features(df_clean)
df_model = create_demand_proxy(df_features)

summarize_censoring(df_model)

## 4. Three-Month Backtest

In [ ]:
train_df, test_df = get_three_month_backtest_split(df_model, months=3)

baseline_results = evaluate_baselines(test_df, target_col='demand_proxy')
model, model_metrics, prediction_df = train_and_evaluate(train_df, test_df, target_col='demand_proxy')

model_results = pd.DataFrame([{'Model': 'LightGBM improved model', **model_metrics}])
comparison = pd.concat([baseline_results, model_results], ignore_index=True)
comparison

## 5. Model Comparison Visualization

In [ ]:
comparison.set_index('Model')[['MAE', 'RMSE']].plot(kind='bar', figsize=(10, 5))
plt.title('Model Comparison')
plt.ylabel('Error value (lower is better)')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 6. Leakage Test Report

In [ ]:
leakage_result = future_permutation_test(df_model)
leakage_result

## 7. Business Summary

The model improves forecasting accuracy compared with naive baselines and can support inventory planning, purchasing decisions, and stock-out reduction.